In [9]:
set.seed(2026); n <- 200                                        # sorteio reproduzivel; 200 anuncios
anuncios <- data.frame(dist  = round(runif(n, 0.1, 5), 1),      # distancia ate a praia (km)
                       fotos = sample(2:20, n, replace = TRUE))  # fotos no anuncio, de 2 a 20
lambda <- exp(1.6 - 0.5 * anuncios$dist + 0.06 * anuncios$fotos)  # a verdade: visitas esperadas em 30 dias
anuncios$visitas <- rpois(n, lambda)                            # visitas observadas em 30 dias
set.seed(11)                                                    # outro sorteio, para o Modulo 5
anuncios$visitas_od <- rpois(n, lambda * rgamma(n, shape = 2, rate = 2))  # contagens mais espalhadas
set.seed(7)                                                     # a base do Modulo 4
an2 <- data.frame(dist  = round(runif(n, 0.1, 5), 1),           # outros 200 anuncios, mesmas colunas
                  fotos = sample(2:20, n, replace = TRUE))       # fotos no anuncio
an2$dias <- round(runif(n, 5, 15) + 9 * an2$dist)               # dias no ar: perto da praia vende antes
an2$visitas <- rpois(n, an2$dias * exp(-1.8 - 0.5 * an2$dist + 0.06 * an2$fotos))  # visitas no periodo
c(sum(anuncios$visitas), sum(anuncios$visitas_od), sum(an2$visitas))  # confira: 836 775 550

[1] 836 775 550

In [10]:
d8 <- data.frame(dist    = c(0.2, 0.5, 1, 1.5, 2, 3, 4, 4.5),  # distancia ate a praia (km)
                 visitas = c(9, 7, 5, 4, 3, 1, 1, 0))           # visitas no mes
reta <- lm(visitas ~ dist, data = d8)                           # o MQO da Aula 4
coef(reta)                                                      # 7,63 e -1,86
predict(reta, data.frame(dist = c(4.5, 5)))                     # -0,73 e -1,66 visitas


(Intercept)        dist 
   7.630133   -1.858746

1          2 
-0.7342258 -1.6635990

In [11]:
m <- glm(visitas ~ dist + fotos, family = poisson, data = anuncios)  # regressao de Poisson
round(coef(m), 4)                    # escala do log: 1,5738  -0,5217  0,0653
round(exp(coef(m)), 4)               # multiplicadores: 4,8248  0,5935  1,0674
novo <- data.frame(dist = 1, fotos = 12)  # um anuncio a 1 km, com 12 fotos
predict(m, novo)                          # 1,835: o LOG das visitas esperadas
predict(m, novo, type = "response")       # 6,27 visitas em 30 dias (a verdade: 6,17)

(Intercept)        dist       fotos 
     1.5738     -0.5217      0.0653

(Intercept)        dist       fotos 
     4.8248      0.5935      1.0674

1 
1.835214

1 
6.266475

In [12]:
sem <- glm(visitas ~ dist + fotos, family = poisson, data = an2)  # ignora os dias no ar
com <- glm(visitas ~ dist + fotos + offset(log(dias)),            # modela visitas por dia
           family = poisson, data = an2)                          # mesma familia
round(exp(coef(sem)), 3)    # dist: 0,832 -- a praia parece valer pouco
round(exp(coef(com)), 3)    # dist: 0,621 -- perto da verdade, 0,607
predict(com, data.frame(dist = 1, fotos = 12, dias = 20), type = "response")  # 3,85 visitas

(Intercept)        dist       fotos 
      2.304       0.832       1.058

(Intercept)        dist       fotos 
      0.160       0.621       1.057

1 
3.849521

In [14]:
mp <- glm(visitas_od ~ dist + fotos, family = poisson, data = anuncios)       # Poisson nas contagens espalhadas
sum(residuals(mp, type = "pearson")^2) / df.residual(mp)                     # phi: 3,37 (a Poisson pede 1)
mq <- glm(visitas_od ~ dist + fotos, family = quasipoisson, data = anuncios)  # mesma conta, erro corrigido
round(cbind(coef(mp), coef(mq)), 4)                                          # coeficientes iguais
round(cbind(summary(mp)$coefficients[, 2],                                   # erros padrao: Poisson
            summary(mq)$coefficients[, 2]), 4)                               # e quase-Poisson: x 1,84

[1] 3.371795

(Intercept),1.4452,1.4452
dist,-0.4444,-0.4444
fotos,0.0609,0.0609


(Intercept),0.1112,0.2041
dist,0.0303,0.0556
fotos,0.0067,0.0124


In [15]:
set.seed(1); dobra <- sample(rep(1:5, length = n))              # as 5 dobras dos laboratorios
rmse_cv <- function(ajusta) sqrt(mean(sapply(1:5, function(j) {  # RMSE da validacao cruzada
  tr <- anuncios[dobra != j, ]; va <- anuncios[dobra == j, ]       # treino e validacao da dobra j
  mean((va$visitas - ajusta(tr, va))^2) })))                       # erro quadratico medio da dobra
rmse_cv(function(tr, va) rep(mean(tr$visitas), nrow(va)))          # linha de base: 3,99
rmse_cv(function(tr, va) predict(lm(visitas ~ dist + fotos, data = tr), va))  # MQO: 2,39
rmse_cv(function(tr, va) predict(glm(visitas ~ dist + fotos, family = poisson,  # Poisson:
                                     data = tr), va, type = "response"))        # 1,76

[1] 3.990111

[1] 2.386974

[1] 1.757921

PARTE D

In [17]:
# --- MODULO 7 - ACHAR A CONTAGEM NO BANCO ---
if (!requireNamespace("readxl", quietly = TRUE)) install.packages("readxl")  # instala readxl, se faltar
dados <- as.data.frame(readxl::read_excel("dataset_final.xlsx"))          # importa a planilha do grupo
stopifnot(ncol(dados) == 10)                                               # confere as 10 colunas do arquivo
names(dados) <- c("ano", "data_obito", "idade", "sexo", "raca",             # nomes simples para as colunas
                  "escolaridade", "municipio", "local_morte", "ocup", "causa_morte")  # completa os nomes
dados$data_obito <- as.Date(dados$data_obito)                               # garante data no formato Date
cat("Linhas do Excel:", nrow(dados), "\n")                                 # informa o tamanho original

chaves <- c("ano", "data_obito", "idade", "sexo", "raca", "escolaridade",  # campos que nao sao ocupacao
            "municipio", "local_morte", "causa_morte")                      # termina a lista de campos
assinatura <- do.call(paste, c(dados[chaves], sep = "\r"))                  # cria assinatura de cada registro
primeira_linha <- c(TRUE, assinatura[-1] != assinatura[-length(assinatura)])  # marca mudancas consecutivas
eventos <- dados[primeira_linha, , drop = FALSE]                            # mantem 1 linha por bloco repetido
cat("Linhas apos aproximacao:", nrow(eventos), "\n")                        # confere numero aproximado

mes_evento <- as.integer(format(eventos$data_obito, "%m"))                 # extrai mes de cada obito
cont <- as.data.frame(table(municipio = factor(eventos$municipio),         # cria combinacoes municipio
                            ano = factor(eventos$ano),                    # inclui todos os anos
                            mes = factor(mes_evento, levels = 1:12)),     # inclui ate meses com zero
                      responseName = "obitos")                              # nome da contagem resultante
primeiro_dia <- as.Date(sprintf("%s-%02d-01", as.character(cont$ano),      # primeiro dia de cada mes
                                 as.integer(as.character(cont$mes))))    # extrai o numero do mes
mes_seguinte <- as.Date(format(primeiro_dia + 32, "%Y-%m-01"))             # primeiro dia do mes seguinte
cont$dias_mes <- as.integer(mes_seguinte - primeiro_dia)                   # exposicao: dias em cada mes
stopifnot(all(cont$dias_mes > 0), all(cont$obitos >= 0))                    # confere exposicao e contagens
cat("Linhas do agregado:", nrow(cont), "\n")                               # esperado: 9 x 5 x 12 = 540
cat("Combinacoes com zero:", sum(cont$obitos == 0), "\n")                   # confirma zeros preservados
summary(cont$obitos)                                                       # descreve a contagem mensal

# --- MODULO 8 - AJUSTAR, LER E CONFERIR ---
m <- glm(obitos ~ municipio + ano + mes + offset(log(dias_mes)),           # Poisson com exposicao mensal
         family = poisson, data = cont)                                     # ajusta o modelo aos agregados
round(exp(coef(m)), 3)                                                      # mostra multiplicadores estimados
phi <- sum(residuals(m, type = "pearson")^2) / df.residual(m)              # mede sobredispersao
cat("Dispersao (phi):", round(phi, 3), "\n")                                 # mostra o indicador
if (phi >= 1.5) {                                                           # sobredispersao importante?
  mq <- glm(obitos ~ municipio + ano + mes + offset(log(dias_mes)),        # quase-Poisson com mesmo modelo
            family = quasipoisson, data = cont)                             # corrige os erros-padrao
  print(summary(mq)$coefficients)                                           # apresenta estimativas e erros
} else {                                                                    # caso a dispersao seja adequada
  print(summary(m)$coefficients)                                            # mostra Poisson original
}                                                                           # encerra o diagnostico

# --- MODULO 9 - VALIDAR E DECIDIR ---
set.seed(1)                                                                 # sorteio reproduzivel das dobras
dobra <- sample(rep(1:5, length.out = nrow(cont)))                          # separa em 5 dobras iguais
rmse_cv <- function(ajusta) {                                               # define a avaliacao de cada modelo
  sqrt(mean(sapply(1:5, function(j) {                                       # RMSE medio das dobras
    tr <- cont[dobra != j, , drop = FALSE]                                   # registros do treino
    va <- cont[dobra == j, , drop = FALSE]                                   # registros da validacao
    mean((va$obitos - ajusta(tr, va))^2)                                     # erro quadratico da dobra
  })))                                                                      # agrega os 5 erros
}                                                                           # encerra a funcao

placar <- c(                                                               # calcula o RMSE de cada modelo
  media = rmse_cv(function(tr, va)                                          # linha de base por taxa diaria
    va$dias_mes * sum(tr$obitos) / sum(tr$dias_mes)),                        # media de obitos ajustada aos dias
  MQO = rmse_cv(function(tr, va)                                            # regressao linear convencional
    predict(lm(obitos ~ municipio + ano + mes + dias_mes, data = tr),       # disponibiliza os mesmos preditores
            newdata = va)),                                                 # calcula previsoes fora do treino
  Poisson = rmse_cv(function(tr, va)                                        # modelo de contagem
    predict(glm(obitos ~ municipio + ano + mes + offset(log(dias_mes)),     # expoe os dias com offset
                family = poisson, data = tr),                               # ajusta apenas na dobra de treino
            newdata = va, type = "response"))                               # previsao em numero de obitos
)                                                                           # encerra a lista de resultados
print(round(placar, 3))                                                     # mostra os 3 RMSE
cat("Menor RMSE:", names(which.min(placar)), "\n")                          # aponta o menor erro

# A avaliacao acima e exploratoria; resultados finais exigem base sem duplicacao.
# Sem dados de populacao por municipio, nao interpretar os coeficientes como
# diferencas de risco de morrer entre cidades (somente volume de obitos).


Linhas do Excel: 474651 
Linhas apos aproximacao: 86683 
Linhas do agregado: 540 
Combinacoes com zero: 0 


   Min. 1st Qu.  Median    Mean 3rd Qu.    Max. 
   20.0    39.0    95.0   160.5   190.2  1052.0 

(Intercept)      municipioCubatão      municipioGuarujá 
                1.124                 1.832                 5.654 
    municipioItanhaém     municipioMongaguá      municipioPeruíbe 
                2.867                 1.136                 1.080 
municipioPraia Grande       municipioSantos  municipioSão Vicente 
                6.767                18.653                 4.702 
              ano2021               ano2022               ano2023 
                1.186                 0.987                 0.909 
              ano2024                  mes2                  mes3 
                0.998                 0.914                 0.972 
                 mes4                  mes5                  mes6 
                1.008                 1.061                 1.019 
                 mes7                  mes8                  mes9 
                0.961                 0.926                 0.864 
                mes10                 mes11                 mes12 
                0.846                 0.900                 0.942

Dispersao (phi): 4.209 
                          Estimate Std. Error     t value      Pr(>|t|)
(Intercept)            0.116563151 0.05305736  2.19692689  2.846956e-02
municipioCubatão       0.605494102 0.05726453 10.57363343  8.744230e-24
municipioGuarujá       1.732404622 0.04996553 34.67199466 6.928577e-137
municipioItanhaém      1.053419477 0.05349005 19.69374683  8.309247e-65
municipioMongaguá      0.127147653 0.06316216  2.01303514  4.462979e-02
municipioPeruíbe       0.076624963 0.06392390  1.19869044  2.311985e-01
municipioPraia Grande  1.912078117 0.04934423 38.74978279 6.736778e-155
municipioSantos        2.926019097 0.04727683 61.89119206 6.249018e-241
municipioSão Vicente   1.547905621 0.05072053 30.51832451 1.188256e-117
ano2021                0.170453493 0.02131841  7.99560015  8.521503e-15
ano2022               -0.013115503 0.02228069 -0.58864896  5.563544e-01
ano2023               -0.095727287 0.02275903 -4.20612379  3.063269e-05
ano2024               -0.001581028 0.022